---
title: Lattice Vibrations Assignment
authors: gvarnavides
date: 2026-06-02
---

In this assignment, we will investigate the vibrational motion of atoms in a crystalline solid and connect these vibrations to quantum statistical physics.

We will model a crystal as a two-dimensional square lattice of atoms connected by harmonic springs.
Although this is a highly simplified model of a real solid, it already captures many important physical ideas:
- collective vibrational motion,
- normal modes,
- phonons,
- dispersion relations,
- and Bose–Einstein statistics.

The assignment also extends the molecular dynamics methods from the first assignment, where you used Velocity Verlet integration to simulate one-dimensional motion.

In [ ]:
import tqdm
import numpy as np
import matplotlib.pyplot as plt
from mpl_toolkits.axes_grid1.axes_divider import make_axes_locatable
from scipy.interpolate import griddata
from lattice_vibrations import make_initial_displacement, velocity_verlet_2d, plot_lattice_snapshot, visualize_orbit, spring_force_2d



### Square Lattice of Springs

We consider a finite two-dimensional square lattice with fixed boundaries.
Each atom is connected to its nearest neighbors by identical harmonic springs.

If an atom is displaced from its equilibrium position, the springs exert restoring forces that tend to pull it back toward equilibrium.
Because all atoms are coupled together, a displacement of one atom produces collective motion throughout the lattice.

For example, for `interior_atoms=15`, the equilibrium (unperturbed) lattice is:

In [ ]:
#| label: phonon_initial_lattice
interior_atoms = 15
num_disp = (interior_atoms+2)**2 * 2

zero_displacements = np.zeros(num_disp)
plot_lattice_snapshot(zero_displacements,interior_atoms=interior_atoms);


Each atom can move in both the horizontal and vertical directions.
The complete lattice configuration is therefore described by a vector of displacements

$$
\mathbf{u}
=
(u_1,v_1,u_2,v_2,\dots),
$$
where $(u_i,v_i)$ denotes the displacement of atom $i$ from its equilibrium position.

### Initial Sinusoidal Perturbations

To excite collective lattice vibrations, we initialize the system with a sinusoidal displacement pattern.

The displacement field is given by

$$
u(x,y)
=
A
\sin(k_x n_x)
\sin(k_y n_y),
$$

where

- $A$ is the displacement amplitude,
- $k_x$ and $k_y$ are wavevectors,
- $n_x$ and $n_y$ are integer lattice coordinates.

These sinusoidal patterns correspond to standing-wave vibrational modes of the lattice.

For example, for $(n_x,n_y)=(3,1)$:

In [ ]:
#| label: phonon_perturbed_lattice
uv_flat, kx,ky = make_initial_displacement(
    nx=3,
    ny=1,
    interior_atoms=interior_atoms
)
plot_lattice_snapshot(uv_flat,interior_atoms=interior_atoms,scale=10);

Different choices of $(n_x,n_y)$ produce different vibrational wavelengths and therefore different oscillation frequencies.

### Velocity Verlet Integration

To evolve the lattice in time, we once again use Velocity Verlet integration.

This is the same numerical integration method used in the first assignment for one-dimensional particle dynamics.
The main difference is that we now evolve many coupled degrees of freedom simultaneously.

You can use the provided  `velocity_verlet_2d` [function](/lattice-vibrations-source-code) as follows:

In [ ]:
#| label: phonon_example_orbit
orbit_qs, orbit_ps, orbit_ts = velocity_verlet_2d(uv_flat,spring_force_2d)

visualize_orbit(
    orbit_qs,
    orbit_ts,
    every=len(orbit_qs)//11,
    scale=10
);

Because the forces are harmonic, the lattice executes periodic collective oscillations known as normal modes.

### Part 3a: 2D Dispersion Relation

In this exercise, we compute the phonon dispersion relation of the 2D square lattice by directly measuring the frequencies of its normal modes.

Each sinusoidal initial displacement corresponds to a mode with wave-vector
$$
\mathbf{k} = (k_x, k_y),
$$
set by the sub-harmonic indices $(n_x,n_y)$.

For each mode:

- Initialize the lattice with the sinusoidal displacement.
- Evolve it using `velocity_verlet_2d`.
- Extract the oscillation period $T$ and compute
    
  $$
  \omega = \frac{2\pi}{T}.
  $$

Compute and plot $\omega(\mathbf{k})$ for all integer values:

$$
n_x, n_y \in \{- (\text{interior\_atoms}+1), \dots, -1, 1, \dots, (\text{interior\_atoms}+1)\},
$$

excluding the zero mode $(n_x,n_y)=(0,0)$.
This produces a discrete 2D map of the vibrational spectrum.

Comment on how the frequency changes with wavelength and identify which modes correspond to long-wavelength (low-$\omega$) excitations.

__Hint__:
You do not need to compute all modes explicitly.  
The system is symmetric under sign changes of $(n_x,n_y)$, so frequencies satisfy
$$
\omega(k_x,k_y) = \omega(-k_x,k_y) = \omega(k_x,-k_y).
$$

### Part 3b: Bose Einstein Occupations

Using the dispersion relation $\omega(\mathbf{k})$, treat each vibrational mode as a quantum harmonic oscillator with [](wiki:Bose_Einstein_statistics)
$$
n(\omega,T) = \frac{1}{\exp(\hbar \omega / k_B T)-1}.
$$

1. For $T \in [0.2, 5.0]$, plot $n(\omega,T)$ as a function of $\omega$.
2. For a fixed temperature (e.g. $T=5.0$), plot $n(\mathbf{k})$.

Comment on why low-frequency modes dominate the occupation and how this changes with temperature.

### Part 3c: Phonon Multiplicity

Compute the phonon multiplicity
$$
g(\omega) = \int \delta(\omega - \omega(\mathbf{k}))\, d\mathbf{k}
$$
using a histogram of your computed dispersion relation.

Plot $g(\omega)$ and identify any sharp features (Van_Hove singularities).

### Part 3d: Quantum Statistics

Compute the internal energy of the lattice using Bose–Einstein statistics:
$$
\langle E \rangle =
\sum_{\mathbf{k}} \hbar \omega(\mathbf{k})
\left(
\frac{1}{\exp(\hbar \omega(\mathbf{k})/k_B T)-1}
+
\frac{1}{2}
\right).
$$

From this, compute and plot the heat capacity
$$
C_v(T) = \frac{d\langle E \rangle}{dT}.
$$

Comment on the approach to the classical Dulong-Petit limit at high temperature and the role of the discrete phonon spectrum at low temperature.